In [1]:
# 📦 Required libraries
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

# ----------------------------------------
# 1. Define input paths for HCB cohort
# ----------------------------------------

nodal_metrics_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/nodal_metrics_with_subjects.csv"
clinical_csv_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/CLINIC_HCB_B.csv"
clinical_label_col = "controls_ms"  # 0 = Control, 1 = EM

# ----------------------------------------
# 2. Load nodal metrics and clinical data
# ----------------------------------------

nodal_df = pd.read_csv(nodal_metrics_path)
clinical_df = pd.read_csv(clinical_csv_path)
clinical_df['id'] = clinical_df['id'].astype(str)
id_to_label = dict(zip(clinical_df['id'], clinical_df[clinical_label_col]))

print(f"✅ Loaded {len(nodal_df['Subject'].unique())} subjects from nodal metrics.")
print(f"✅ Loaded {len(id_to_label)} clinical labels.")

# ----------------------------------------
# 3. Define brain graph structure (fully connected)
# ----------------------------------------

N = 76  # number of brain regions
metric_cols = ["degree", "strength", "betweenness", "closeness", "eigenvector"]
layers = ["FA", "GM", "rsfMRI"]
data_lists = {}

# Fully connected undirected graph (bidirectional)
edge_index = torch.combinations(torch.arange(N), r=2).T
edge_index = torch.cat([edge_index, edge_index[[1, 0], :]], dim=1)

# ----------------------------------------
# 4. Build PyTorch Geometric data_list per layer
# ----------------------------------------

for layer in layers:
    print(f"\n🧠 Processing layer: {layer}")

    # Filter nodal metrics for this layer
    layer_df = nodal_df[nodal_df["Layer"] == layer]

    # Create graph per subject
    data_list = []
    for subj in layer_df["Subject"].unique():
        subj_data = layer_df[layer_df["Subject"] == subj].sort_values("Node")

        if subj not in id_to_label:
            continue  # skip subjects without label

        x = torch.tensor(subj_data[metric_cols].values, dtype=torch.float32)
        y = torch.tensor([id_to_label[subj]], dtype=torch.long)

        data = Data(x=x, edge_index=edge_index.clone(), y=y)
        data.subject_id = subj
        data_list.append(data)

    data_lists[layer] = data_list
    print(f"✅ Created {len(data_list)} graph objects for layer: {layer}")


✅ Loaded 163 subjects from nodal metrics.
✅ Loaded 165 clinical labels.

🧠 Processing layer: FA
✅ Created 163 graph objects for layer: FA

🧠 Processing layer: GM
✅ Created 155 graph objects for layer: GM

🧠 Processing layer: rsfMRI
✅ Created 163 graph objects for layer: rsfMRI


In [5]:
# 📦 Required libraries
from nilearn import plotting
import matplotlib.pyplot as plt
import os

# ----------------------------------------
# 1. Load MNI coordinates if not already loaded
# ----------------------------------------

if 'mni_coords' not in globals():
    mni_coords_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/Node_mindboggle_default.node"
    with open(mni_coords_path, "r") as f:
        lines = f.readlines()
    mni_coords = []
    for line in lines:
        parts = line.strip().split('\t')
        try:
            coord = list(map(float, parts[:3]))
            mni_coords.append(coord)
        except ValueError:
            continue
    mni_coords = np.array(mni_coords)

# ----------------------------------------
# 2. Define output directory (HCB version)
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 3. Loop over layers and metrics
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

for layer in data_lists.keys():
    print(f"\n🧠 Generating plots for layer: {layer}")

    data_list = data_lists[layer]
    control_graphs = [data for data in data_list if data.y.item() == 0]
    ms_graphs = [data for data in data_list if data.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    for metric_index, metric_name in enumerate(metric_names):
        # Compute Δ = Control − MS
        mean_control = control_features[:, :, metric_index].mean(dim=0).numpy()
        mean_ms = ms_features[:, :, metric_index].mean(dim=0).numpy()
        delta = mean_control - mean_ms

        # Empty adjacency (used just for plotting node positions)
        empty_adj = np.zeros((76, 76))

        # Plot node-level Δ metric
        title = f"{layer} - Δ {metric_name} (Control − MS)\nYellow: ↑ Control | Purple: ↑ MS"
        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,                                           
                                           title=title)

        # Save plot
        filename = f"{layer}_delta_{metric_name.lower().replace(' ', '_')}_control_minus_ms.png"
        output_file = os.path.join(output_dir, filename)
        plt.savefig(output_file, dpi=300)
        plt.close()
        print(f"✅ Saved: {output_file}")



🧠 Generating plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_eigenvector_control_minus_ms.png

🧠 Gen

In [4]:
# 📂 Define output directory (HCB version)
output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# 📈 Define metric names
metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

# 🔁 Loop over each layer and compute Δ (Control − MS)
for layer, data_list in data_lists.items():
    print(f"\n📊 Computing nodal deltas for layer: {layer}")
    
    control_graphs = [data for data in data_list if data.y.item() == 0]
    ms_graphs = [data for data in data_list if data.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    all_deltas = []

    for metric_index, metric_name in enumerate(metric_names):
        mean_control = control_features[:, :, metric_index].mean(dim=0).numpy()
        mean_ms = ms_features[:, :, metric_index].mean(dim=0).numpy()
        delta = mean_control - mean_ms

        for node in range(76):
            all_deltas.append({
                "Layer": layer,
                "Node": node,
                "Metric": metric_name,
                "Mean_Control": mean_control[node],
                "Mean_MS": mean_ms[node],
                "Delta_Control_minus_MS": delta[node]
            })

    # 💾 Save node-level deltas to CSV
    delta_df = pd.DataFrame(all_deltas)
    csv_path = os.path.join(output_dir, f"{layer}_nodal_deltas_control_minus_ms.csv")
    delta_df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved: {csv_path}")



📊 Computing nodal deltas for layer: FA
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_nodal_deltas_control_minus_ms.csv

📊 Computing nodal deltas for layer: GM
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\GM_nodal_deltas_control_minus_ms.csv

📊 Computing nodal deltas for layer: rsfMRI
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_nodal_deltas_control_minus_ms.csv


In [6]:
# ----------------------------------------
# 1. Define output directory for HCB
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Subjects with RRMS and SPMS labels (HCB cohort)
# ----------------------------------------

subject_labels = nodal_df[["Subject", "mstype_label"]].drop_duplicates()
rrms_subjects = subject_labels[subject_labels["mstype_label"] == "RRMS"]["Subject"].tolist()
spms_subjects = subject_labels[subject_labels["mstype_label"] == "SPMS"]["Subject"].tolist()

# ----------------------------------------
# 3. Generate RRMS − SPMS plots
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

for layer, data_list in data_lists.items():
    print(f"\n🧠 Generating RRMS − SPMS plots for layer: {layer}")

    rrms_graphs = [g for g in data_list if hasattr(g, 'subject_id') and g.subject_id in rrms_subjects]
    spms_graphs = [g for g in data_list if hasattr(g, 'subject_id') and g.subject_id in spms_subjects]

    # Skip if one group is missing
    if not rrms_graphs or not spms_graphs:
        print(f"⚠️ Skipping layer {layer} (missing RRMS or SPMS subjects)")
        continue

    rrms_features = torch.stack([g.x for g in rrms_graphs])
    spms_features = torch.stack([g.x for g in spms_graphs])

    for i, metric in enumerate(metric_names):
        mean_rrms = rrms_features[:, :, i].mean(dim=0).numpy()
        mean_spms = spms_features[:, :, i].mean(dim=0).numpy()
        delta = mean_rrms - mean_spms  # RRMS − SPMS

        title = f"{layer} - Δ {metric} per Node (RRMS − SPMS)\nYellow = ↑ RRMS | Purple = ↑ SPMS"
        empty_adj = np.zeros((76, 76))

        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,                                           
                                           title=title)

        filename = f"{layer}_delta_{metric.lower()}_rrms_minus_spms.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300)
        plt.close()
        print(f"✅ Saved: {filepath}")



🧠 Generating RRMS − SPMS plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_eigenvector_rrms_minus_spms.png

In [7]:
# ----------------------------------------
# 1. Output directory for HCB
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Filter EDSS groups for HCB
# ----------------------------------------

subject_labels = nodal_df[["Subject", "edss_group"]].drop_duplicates()
mild_subjects = subject_labels[subject_labels["edss_group"] == "0–2 (Mild)"]["Subject"].tolist()
severe_subjects = subject_labels[subject_labels["edss_group"] == "6.5–9 (Severe)"]["Subject"].tolist()

# ----------------------------------------
# 3. Metric names
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

# ----------------------------------------
# 4. Loop over layers and plot
# ----------------------------------------

for layer, data_list in data_lists.items():
    print(f"\n🧠 Generating EDSS Mild − Severe plots for layer: {layer}")

    mild_graphs = [g for g in data_list if hasattr(g, 'subject_id') and g.subject_id in mild_subjects]
    severe_graphs = [g for g in data_list if hasattr(g, 'subject_id') and g.subject_id in severe_subjects]

    # Skip if one group is missing
    if not mild_graphs or not severe_graphs:
        print(f"⚠️ Skipping layer {layer} (missing Mild or Severe subjects)")
        continue

    mild_features = torch.stack([g.x for g in mild_graphs])
    severe_features = torch.stack([g.x for g in severe_graphs])

    for i, metric in enumerate(metric_names):
        mean_mild = mild_features[:, :, i].mean(dim=0).numpy()
        mean_severe = severe_features[:, :, i].mean(dim=0).numpy()
        delta = mean_mild - mean_severe

        title = f"{layer} - Δ {metric} per Node (EDSS Mild − Severe)\nYellow = ↑ Mild | Purple = ↑ Severe"
        empty_adj = np.zeros((76, 76))

        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,                                           
                                           title=title)

        filename = f"{layer}_delta_{metric.lower()}_edss_mild_minus_severe.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300)
        plt.close()
        print(f"✅ Saved: {filepath}")



🧠 Generating EDSS Mild − Severe plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_de

In [9]:
# 📦 Required libraries
import os
import torch
import random
import numpy as np
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# ----------------------------------------
# 1. Set random seeds
# ----------------------------------------
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ----------------------------------------
# 2. Define GNN architectures
# ----------------------------------------
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=1)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

# ----------------------------------------
# 3. Training and evaluation
# ----------------------------------------
def train(model, loader, optimizer, device):
    model.train()
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()

def test(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            prob = F.softmax(out, dim=1)
            pred = prob.argmax(dim=1)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            y_prob.extend(prob[:, 1].cpu().numpy())
    return y_true, y_pred, y_prob

# ----------------------------------------
# 4. Cross-validation loop with confusion matrix
# ----------------------------------------
def run_cross_validation(ModelClass, model_name, data_list, layer, output_dir, k=5, hidden_dim=32, epochs=100):
    print(f"\n🔁 Running {model_name} for layer {layer} with {k}-Fold Stratified CV")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    labels = [data.y.item() for data in data_list]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]

        train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=16)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelClass(in_channels=5, hidden_channels=hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        for epoch in range(epochs):
            train(model, train_loader, optimizer, device)

        y_true, y_pred, y_prob = test(model, test_loader, device)

        # Plot confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Control", "MS"])
        disp.plot(cmap="Blues")
        plt.title(f"{model_name} - {layer} - Fold {fold+1}")
        cm_path = os.path.join(output_dir, f"{layer}_{model_name}_fold{fold+1}_confmat.png")
        plt.savefig(cm_path, dpi=150)
        plt.close()
        print(f"✅ Saved confusion matrix: {cm_path}")

        metrics = {
            "Model": model_name,
            "Layer": layer,
            "Fold": fold + 1,
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "AUC": roc_auc_score(y_true, y_prob)
        }
        fold_results.append(metrics)
        print(f"📊 Fold {fold+1} - Acc: {metrics['Accuracy']:.3f}, F1: {metrics['F1']:.3f}, AUC: {metrics['AUC']:.3f}")

    df = pd.DataFrame(fold_results)
    avg = df.iloc[:, 3:].mean().to_dict()
    avg_row = {"Model": model_name, "Layer": layer, "Fold": "Average", **avg}
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n📋 Results for {model_name} - {layer}:")
    print(df.round(3).to_string(index=False))

    folds_path = os.path.join(output_dir, f"{layer}_GNN_metrics_folds.csv")
    df.to_csv(folds_path, index=False, mode='a', header=not os.path.exists(folds_path))
    print(f"✅ Appended results to {folds_path}")

# ----------------------------------------
# 5. Run for all models and layers - HCB
# ----------------------------------------
output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

for layer, data_list in data_lists.items():
    run_cross_validation(GCN, "GCN", data_list, layer, output_dir)
    run_cross_validation(GraphSAGE, "GraphSAGE", data_list, layer, output_dir)
    run_cross_validation(GAT, "GAT", data_list, layer, output_dir)



🔁 Running GCN for layer FA with 5-Fold Stratified CV
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold1_confmat.png
📊 Fold 1 - Acc: 0.879, F1: 0.935, AUC: 0.716
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold2_confmat.png
📊 Fold 2 - Acc: 0.879, F1: 0.935, AUC: 0.716
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold3_confmat.png
📊 Fold 3 - Acc: 0.879, F1: 0.935, AUC: 0.276
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_HCB/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold4_confmat.png
📊 Fold 4 - Acc: 0.906, F1: 0.951, A